# Macroeconomic Time-Series Modeling and Forecasting

## VAR, VARX, and Machine-Learning Forecast Comparison

This notebook is structured as an academic econometrics/data-science project. It studies a monthly U.S. macroeconomic system using real FRED data, with VAR and VARX models as the main econometric framework and machine-learning methods used only as forecast benchmarks.

**Research focus:** Can inflation dynamics be explained and forecast using monetary policy, labor market conditions, industrial production, money supply, and consumer sentiment?


## 1. Motivation and Problem Formulation

Macroeconomic forecasting is central to policy and private-sector decision-making. Inflation forecasts affect monetary policy, wage bargaining, financial-market expectations, business planning, and household decisions. Central banks monitor inflation jointly with unemployment, output, money, interest rates, and expectations because inflation is not an isolated process; it is part of a dynamic macroeconomic system.

This project asks whether a system of U.S. monthly macroeconomic indicators can explain and forecast inflation. The core forecasting target is monthly log CPI inflation. The explanatory system contains:

- **Federal Funds Rate:** monetary-policy stance and short-term interest-rate channel.
- **Unemployment Rate:** labor market slack and Phillips-curve pressure.
- **Industrial Production:** real activity and demand-side pressure.
- **M2 Money Supply:** liquidity and monetary aggregates.
- **Consumer Sentiment:** expectations/confidence channel.

The problem is both statistical and economic: a model should forecast reasonably well, but it should also support interpretable economic analysis through Granger causality, impulse responses, and forecast error variance decomposition.

## 2. International Experience and Literature Review

International macroeconomic forecasting commonly combines structural economic reasoning with empirical time-series models. Since Sims (1980), **Vector Autoregressions (VARs)** have been widely used to model macro variables as jointly endogenous systems. Central banks and policy institutions often use VARs, Bayesian VARs, factor-augmented VARs, DSGE-VAR hybrids, and forecast-combination systems to study monetary transmission and produce scenario forecasts.

Key methods used in this project:

- **VAR:** treats all selected macro variables as jointly endogenous and captures lagged dynamic interactions.
- **VARX:** extends VAR by conditioning the endogenous system on exogenous or policy/scenario variables. Forecasts are conditional on assumed future paths for those exogenous variables.
- **Granger causality:** tests whether lagged values of one variable improve prediction of another variable, without proving structural causality.
- **Impulse Response Functions (IRF):** trace the dynamic response of the system to structural shocks.
- **Forecast Error Variance Decomposition (FEVD):** decomposes forecast uncertainty into contributions from each structural shock.
- **Machine learning forecasting:** Random Forest, Gradient Boosting, and regularized regression can capture nonlinearities or high-dimensional lag patterns, but usually sacrifice economic interpretability.

Selected references and methodological anchors:

- Sims, C. A. (1980), *Macroeconomics and Reality*, Econometrica. DOI: https://doi.org/10.2307/1912017
- Granger, C. W. J. (1969), *Investigating Causal Relations by Econometric Models and Cross-spectral Methods*, Econometrica. DOI: https://doi.org/10.2307/1912791
- Lütkepohl, H. (2005), *New Introduction to Multiple Time Series Analysis*, Springer.
- Bernanke, Boivin, and Eliasz (2005), *Measuring the Effects of Monetary Policy: A Factor-Augmented VAR Approach*, QJE. DOI: https://doi.org/10.1162/0033553053327452
- Stock and Watson (2002), *Macroeconomic Forecasting Using Diffusion Indexes*, JBES. DOI: https://doi.org/10.1198/073500102317351921
- Diebold and Mariano (1995), *Comparing Predictive Accuracy*, JBES. DOI: https://doi.org/10.1080/07350015.1995.10524599
- FRED API documentation: https://fred.stlouisfed.org/docs/api/fred/


In [1]:
from pathlib import Path
import os

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
TABLE_DIR = BASE_DIR / "outputs" / "tables"
FIGURE_DIR = BASE_DIR / "outputs" / "figures"
os.environ.setdefault("MPLCONFIGDIR", str(BASE_DIR / ".matplotlib"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)

raw = pd.read_csv(DATA_DIR / "raw_fred_macro.csv", parse_dates=["date"], index_col="date")
model_df = pd.read_csv(DATA_DIR / "academic_model_data.csv", parse_dates=["date"], index_col="date")
raw.shape, model_df.shape

((376, 6), (375, 6))

## 3. Relevant Data Collection

Data are collected from the Federal Reserve Economic Data (FRED) API. Monthly frequency is used because all selected macro variables are available monthly and because monetary-policy and macro-cycle dynamics are usually studied at monthly or quarterly frequencies.

The dataset is aligned to monthly-start frequency. Missing observations are forward-filled only after monthly alignment, which is appropriate for occasional release-calendar gaps but should not be interpreted as creating new economic information. After transformation, the model dataset contains 375 monthly observations.

In [2]:
variable_dictionary = pd.read_csv(TABLE_DIR / "academic_variable_dictionary.csv")
variable_dictionary

,variable,fred_id,economic_role,expected_use
0,CPI,CPIAUCSL,Price level used to construct inflation.,Target transformation
1,UNRATE,UNRATE,Labor market slack and cyclical pressure.,Endogenous macro state
2,FEDFUNDS,FEDFUNDS,Monetary-policy stance and short-term interest...,"Endogenous in VAR, exogenous in VARX"
3,INDPRO,INDPRO,Real production and business-cycle activity.,Growth transformation
4,M2,M2SL,Broad money supply and liquidity conditions.,Growth transformation
5,UMCSENT,UMCSENT,Household expectations and confidence channel.,Change transformation


In [3]:
missing_values = pd.read_csv(TABLE_DIR / "academic_missing_values.csv")
missing_values

,Unnamed: 0,missing_values
0,CPI,0
1,UNRATE,0
2,FEDFUNDS,0
3,INDPRO,0
4,M2,0
5,UMCSENT,0


## 4. Exploratory Data Analysis

EDA serves two purposes. First, it identifies economic episodes such as the 2008 financial crisis, COVID shock, inflation surge, and monetary tightening. Second, it checks statistical properties such as persistence, dispersion, nonlinear relationships, and possible structural breaks.

### 4.1 Raw Time-Series Visualization

Raw CPI and M2 are trending level variables, while interest rates, unemployment, industrial production, and sentiment show strong cyclical behavior.

![Raw FRED series](outputs/figures/academic_01_raw_series.png)

In [4]:
raw_summary = pd.read_csv(TABLE_DIR / "academic_raw_summary_statistics.csv", index_col=0)
raw_summary.round(3)

,count,mean,std,min,25%,50%,75%,max
CPI,376.0,222.741,47.714,150.500,181.425,218.946,251.067,332.407
UNRATE,376.0,5.513,1.805,3.400,4.300,5.000,5.925,14.800
FEDFUNDS,376.0,2.564,2.233,0.050,0.178,1.910,5.070,6.540
INDPRO,376.0,94.621,7.954,71.188,90.639,97.664,100.845,104.100
M2,376.0,10655.839,5990.244,3489.900,5745.350,8703.300,14146.550,22804.500
UMCSENT,376.0,84.471,14.861,49.800,73.125,87.900,95.825,112.000


### 4.2 Transformed Model Variables

For econometric modeling, CPI, industrial production, and M2 are converted into log growth rates. Consumer sentiment is differenced. Rates are kept in levels after stationarity checks.

![Transformed model variables](outputs/figures/academic_02_transformed_series.png)

![Rolling means](outputs/figures/academic_03_rolling_means.png)

In [5]:
model_summary = pd.read_csv(TABLE_DIR / "academic_model_summary_statistics.csv", index_col=0)
model_summary.round(3)

,count,mean,std,min,25%,50%,75%,max
FEDFUNDS,375.0,2.556,2.230,0.050,0.175,1.910,5.065,6.540
INF,375.0,0.211,0.282,-1.786,0.068,0.209,0.328,1.367
UNRATE,375.0,5.513,1.808,3.400,4.300,5.000,5.950,14.800
INDPRO_GROWTH,375.0,0.097,1.085,-14.142,-0.237,0.136,0.514,6.347
M2_GROWTH,375.0,0.500,0.556,-1.404,0.280,0.451,0.641,6.240
SENTIMENT_CHANGE,375.0,-0.127,4.108,-17.300,-2.500,-0.200,2.450,11.200


### 4.3 Correlation, Scatter, and Distribution Analysis

The correlation matrix and scatter plots help identify contemporaneous relationships, but they do not establish causality. Dynamic models and lagged tests are required because macroeconomic effects often occur with delay.

![Correlation matrix](outputs/figures/academic_04_correlation_matrix.png)

![Scatter matrix](outputs/figures/academic_05_scatter_matrix.png)

![Distributions](outputs/figures/academic_06_distributions.png)

In [6]:
corr = pd.read_csv(TABLE_DIR / "academic_model_correlation_matrix.csv", index_col=0)
corr.round(3)

,FEDFUNDS,INF,UNRATE,INDPRO_GROWTH,M2_GROWTH,SENTIMENT_CHANGE
FEDFUNDS,1.000,0.072,-0.570,0.061,-0.175,-0.023
INF,0.072,1.000,-0.158,0.218,-0.217,-0.096
UNRATE,-0.570,-0.158,1.000,-0.061,0.342,0.001
INDPRO_GROWTH,0.061,0.218,-0.061,1.000,-0.397,0.153
M2_GROWTH,-0.175,-0.217,0.342,-0.397,1.000,-0.232
SENTIMENT_CHANGE,-0.023,-0.096,0.001,0.153,-0.232,1.000


### 4.4 ACF/PACF and Structural Break Inspection

The ACF and PACF of inflation motivate lagged dynamic modeling. Structural break inspection is important because the 2008 crisis and COVID period can distort estimates if ignored.

![Inflation ACF PACF](outputs/figures/academic_07_inflation_acf_pacf.png)

![Structural break inspection](outputs/figures/academic_08_structural_break_inspection.png)

## 5. Model Data Preparation

The preprocessing decisions follow standard time-series logic:

- Positive trending level variables are log transformed and differenced to approximate growth rates.
- Inflation is measured as monthly log CPI growth multiplied by 100.
- Industrial production and M2 are also modeled as monthly log growth rates.
- Consumer sentiment is differenced because it is an index and can be non-stationary in level.
- The unemployment rate and federal funds rate are kept in levels after ADF testing supports stationarity in this sample.
- Step dummies are added for 2008 and COVID to control intercept shifts associated with major structural breaks.
- The final 36 months are reserved for out-of-sample forecast evaluation.

In [7]:
raw_adf = pd.read_csv(TABLE_DIR / "academic_raw_adf_tests.csv")
raw_adf[["variable", "p_value", "stationary_at_5pct"]]

,variable,p_value,stationary_at_5pct
0,CPI,0.9985,False
1,UNRATE,0.0223,True
2,FEDFUNDS,0.0488,True
3,INDPRO,0.0492,True
4,M2,0.9935,False
5,UMCSENT,0.6951,False


In [8]:
cointegration = pd.read_csv(TABLE_DIR / "academic_pairwise_cointegration.csv")
cointegration

,left_variable,right_variable,residual_adf_p_value,cointegrated_at_5pct
0,log_CPI,log_M2,0.2065,False
1,log_CPI,UMCSENT,0.3650,False
2,log_M2,UMCSENT,0.4267,False


In [9]:
transformed_adf = pd.read_csv(TABLE_DIR / "academic_transformed_adf_tests.csv")
transformed_adf[["variable", "p_value", "stationary_at_5pct"]]

,variable,p_value,stationary_at_5pct
0,FEDFUNDS,2.8806e-02,True
1,INF,6.2963e-03,True
2,UNRATE,2.2636e-02,True
3,INDPRO_GROWTH,8.3255e-27,True
4,M2_GROWTH,1.0078e-06,True
5,SENTIMENT_CHANGE,2.1818e-21,True


In [10]:
break_dummies = pd.read_csv(DATA_DIR / "academic_break_dummies.csv", parse_dates=["date"], index_col="date")
break_dummies.tail()

,D_2008,D_COVID
date,,
2025-12-01,1,1
2026-01-01,1,1
2026-02-01,1,1
2026-03-01,1,1
2026-04-01,1,1


## 6. Econometric Model Construction

### 6.1 VAR and VARX Lag Selection

A VAR models each variable as a function of its own lags and the lags of all other endogenous variables. This is appropriate when macro variables are jointly determined. The baseline VAR also includes two exogenous crisis dummies.

VARX uses the same dynamic structure for a smaller endogenous block but conditions forecasts on externally supplied exogenous paths. In this project the exogenous paths are the federal funds rate, sentiment change, and crisis dummies. This makes VARX especially useful for conditional and scenario forecasting, but it also means the forecast is only as credible as the assumed future exogenous path.

Lag order is selected using information criteria. The official baseline uses 4 monthly lags because AIC selects 4 for the full VAR, monthly macro dynamics often need several lags, and the fixed baseline keeps the notebook reproducible. The dashboard later allows sensitivity analysis over lag order, train/test split, variables, and VAR versus VARX specification.


In [11]:
var_lag_table = pd.read_csv(TABLE_DIR / "academic_var_lag_selection.csv")
varx_lag_table = pd.read_csv(TABLE_DIR / "academic_varx_lag_selection.csv")
print("VAR lag selection")
display(var_lag_table.round(4))
print("VARX lag selection")
display(varx_lag_table.round(4))


VAR lag selection


,lag,AIC,BIC,HQIC,FPE
0,0,0.4717,0.6648,0.5484,1.6027
1,1,-7.5706,-6.9912,-7.3403,0.0005
2,2,-8.3444,-7.3789,-7.9606,0.0002
3,3,-8.4758,-7.1240,-7.9385,0.0002
4,4,-8.5428,-6.8048,-7.8519,0.0002
5,5,-8.5141,-6.3898,-7.6697,0.0002
6,6,-8.4811,-5.9706,-7.4832,0.0002
7,7,-8.4509,-5.5542,-7.2995,0.0002
8,8,-8.4449,-5.1620,-7.1400,0.0002
9,9,-8.3856,-4.7165,-6.9271,0.0002


VARX lag selection


,lag,AIC,BIC,HQIC,FPE
0,0,-3.2519,-3.0374,-3.1667,0.0387
1,1,-6.6571,-6.2709,-6.5036,0.0013
2,2,-6.8599,-6.3020,-6.6381,0.0010
3,3,-6.9356,-6.2060,-6.6456,0.0010
4,4,-7.0001,-6.0989,-6.6418,0.0009
5,5,-6.9874,-5.9146,-6.5609,0.0009
6,6,-6.9698,-5.7253,-6.4751,0.0009
7,7,-6.9481,-5.5319,-6.3851,0.0010
8,8,-6.9341,-5.3463,-6.3030,0.0010
9,9,-6.9180,-5.1586,-6.2187,0.0010


### 6.2 Final Model Architecture

The final project uses a layered architecture rather than a single model for every purpose.

**Primary system model: VAR(4).** The baseline reduced-form model is a VAR with all transformed macro variables treated as endogenous:

`FEDFUNDS, INF, UNRATE, INDPRO_GROWTH, M2_GROWTH, SENTIMENT_CHANGE`

The model includes `D_2008` and `D_COVID` as exogenous break controls. The lag order is **4 monthly lags**, selected by minimum AIC over candidate lag orders for the full VAR. BIC/HQIC and residual autocorrelation robustness are reported as checks.

**Conditional forecasting model: VARX(4).** For conditional forecasting, the endogenous block is:

`INF, UNRATE, INDPRO_GROWTH, M2_GROWTH`

The exogenous block is:

`FEDFUNDS, SENTIMENT_CHANGE, D_2008, D_COVID`

This VARX specification treats policy, sentiment, and break dummies as externally conditioned paths. Economically, this is useful for scenario analysis: the researcher can ask what happens to inflation and real activity given a path for policy rates and sentiment. The reported baseline keeps 4 lags for comparability with the VAR; the separate VARX lag-selection table and dashboard model lab show how alternative lag choices affect results.

**Structural interpretation layer: recursive SVAR.** IRF and FEVD analysis use a Cholesky decomposition of the reduced-form VAR residual covariance matrix. This is not a separate estimated structural model; it is a recursive identification layer imposed on the VAR.


In [12]:
architecture = pd.read_csv(TABLE_DIR / "academic_final_model_architecture.csv")
architecture

,component,model,endogenous_variables,exogenous_variables,lag_order,selection_or_role,purpose
0,Primary reduced-form model,VAR,"FEDFUNDS, INF, UNRATE, INDPRO_GROWTH, M2_GROWT...","D_2008, D_COVID",4,Lag order selected by minimum AIC over candida...,"Main system model for dynamic interactions, Gr..."
1,Conditional forecasting model,VARX,"INF, UNRATE, INDPRO_GROWTH, M2_GROWTH","FEDFUNDS, SENTIMENT_CHANGE, D_2008, D_COVID",4,Uses the same lag order as the baseline VAR fo...,Conditional forecast model where policy/sentim...
2,Structural interpretation layer,Recursive SVAR via Cholesky decomposition of V...,"FEDFUNDS, INF, UNRATE, INDPRO_GROWTH, M2_GROWT...","D_2008, D_COVID",4,Uses the reduced-form VAR ordering for Cholesk...,Impulse response and FEVD analysis under expli...


#### Cholesky Identification and Variable Ordering

The recursive Cholesky ordering is:

`FEDFUNDS -> INF -> UNRATE -> INDPRO_GROWTH -> M2_GROWTH -> SENTIMENT_CHANGE`

This ordering is motivated by three arguments:

- **Policy transmission logic:** the federal funds rate is placed first as a fast-moving policy/financial variable that can contemporaneously transmit to prices, labor markets, output, money, and expectations.
- **Speed of adjustment:** financial and price variables are allowed to adjust earlier; labor markets and real production are slower; sentiment is placed last so it can contemporaneously absorb macro news.
- **Granger evidence:** the federal funds rate has significant predictive content for inflation and several real/monetary variables, supporting its central role in the dynamic system.

The imposed contemporaneous restrictions are asymmetric. A variable ordered earlier does not contemporaneously respond to shocks from variables ordered later, while later variables may respond contemporaneously to earlier shocks. Therefore, IRFs are conditional on this ordering. A different ordering could change the short-run impulse responses, especially near horizon 0.

In [13]:
ordering = pd.read_csv(TABLE_DIR / "academic_cholesky_ordering.csv")
ordering

,order,variable,contemporaneous_assumption,economic_motivation
0,1,FEDFUNDS,Policy rate can contemporaneously affect all l...,Monetary policy is a fast-moving financial/pol...
1,2,INF,Inflation reacts within the month to policy sh...,Prices are central to the policy objective; in...
2,3,UNRATE,Labor-market conditions react contemporaneousl...,Unemployment adjusts more slowly than financia...
3,4,INDPRO_GROWTH,Industrial production growth can react to poli...,Production is a real-activity measure that res...
4,5,M2_GROWTH,Money growth reacts contemporaneously to the v...,"Broad money adjusts through banking, portfolio..."
5,6,SENTIMENT_CHANGE,Sentiment is allowed to react contemporaneousl...,Survey sentiment is fast-moving but often refl...


### 6.3 VAR Diagnostics: Stability, Serial Correlation, and Residual Dependence

A serious VAR diagnostic workflow cannot rely only on numerical tests. The project therefore combines formal tests with visual residual diagnostics.

The stability condition is checked using inverse roots of the characteristic polynomial. Residual serial correlation is checked with Durbin-Watson, Ljung-Box tests, residual ACF plots, and a full residual ACF/CCF matrix. In the residual ACF plots, bars outside the approximate 95% confidence bands indicate remaining serial dependence. In real macroeconomic data, perfect residual whitening is difficult, but the objective is to reduce autocorrelation as much as reasonably possible without overfitting.

The AIC-selected VAR(4) is stable. Individual Ljung-Box tests are mostly acceptable, but several residual ACF plots still show isolated lags outside confidence bounds and the multivariate whiteness test rejects. This is an important limitation and suggests that alternative lag lengths, richer break modeling, seasonal controls, or additional exogenous variables could be considered in further work.

![VAR residuals](outputs/figures/academic_var_residual_plots.png)

![VAR residual ACF](outputs/figures/academic_var_residual_acf.png)

![VAR residual ACF and CCF matrix](outputs/figures/academic_var_residual_acf_ccf_matrix.png)

![VAR residual correlation](outputs/figures/academic_var_residual_correlation_heatmap.png)

![VAR stability roots](outputs/figures/academic_var_stability_roots.png)

In [14]:
var_diag = pd.read_csv(TABLE_DIR / "academic_var_residual_diagnostics.csv")
var_diag.round(4)

,variable,durbin_watson,ljung_box_p_value_lag12,white_noise_at_5pct,arch_lm_p_value_lag12,arch_effects_at_5pct,stable_system,multivariate_whiteness_p_value
0,FEDFUNDS,2.0246,0.4100,True,0.9815,False,True,0.0
1,INF,2.0053,0.2936,True,0.0001,True,True,0.0
2,UNRATE,2.0058,0.7680,True,1.0000,False,True,0.0
3,INDPRO_GROWTH,2.0129,0.1701,True,0.1849,False,True,0.0
4,M2_GROWTH,1.9789,0.3365,True,0.0000,True,True,0.0
5,SENTIMENT_CHANGE,2.0837,0.2736,True,0.0922,False,True,0.0


#### Visual Autocorrelation Summary

The table below summarizes whether residual ACF values exceed approximate 95% bounds. A small number of exceedances is common in macroeconomic systems, but repeated exceedances would indicate that the lag structure or specification may still be incomplete.

In [15]:
var_acf_summary = pd.read_csv(TABLE_DIR / "academic_var_residual_acf_summary.csv")
var_acf_summary.round(4)

,equation,max_abs_acf_lag_1_to_24,confidence_bound_approx,num_lags_outside_bound,any_lag_outside_bound
0,FEDFUNDS,0.1288,0.1018,1,True
1,INF,0.1338,0.1018,2,True
2,UNRATE,0.0833,0.1018,0,False
3,INDPRO_GROWTH,0.1385,0.1018,2,True
4,M2_GROWTH,0.1265,0.1018,2,True
5,SENTIMENT_CHANGE,0.1479,0.1018,1,True


In [16]:
var_ccf_summary = pd.read_csv(TABLE_DIR / "academic_var_residual_ccf_summary.csv")
var_ccf_summary.sort_values("num_lags_outside_bound", ascending=False).head(12).round(4)

,source_residual,target_residual,max_abs_ccf_lag_1_to_24,confidence_bound_approx,num_lags_outside_bound,any_lag_outside_bound
8,M2_GROWTH,INF,0.1225,0.1018,3,True
3,M2_GROWTH,FEDFUNDS,0.1133,0.1018,2,True
6,UNRATE,INF,0.1092,0.1018,2,True
17,UNRATE,INDPRO_GROWTH,0.1535,0.1018,2,True
15,FEDFUNDS,INDPRO_GROWTH,0.1728,0.1018,2,True
14,SENTIMENT_CHANGE,UNRATE,0.1505,0.1018,2,True
18,M2_GROWTH,INDPRO_GROWTH,0.1261,0.1018,2,True
27,UNRATE,SENTIMENT_CHANGE,0.1133,0.1018,2,True
28,INDPRO_GROWTH,SENTIMENT_CHANGE,0.1520,0.1018,2,True
16,INF,INDPRO_GROWTH,0.1466,0.1018,1,True


#### Lag-Robustness Check for Residual Autocorrelation

AIC selects 4 lags, but residual autocorrelation exceedances decline further at some higher lag orders. This illustrates the standard tradeoff: more lags may whiten residuals better, while fewer lags preserve degrees of freedom and reduce overfitting. The project keeps VAR(4) as the baseline because it is selected by AIC and remains stable, but the robustness table is reported transparently.

In [17]:
lag_autocorr = pd.read_csv(TABLE_DIR / "academic_var_lag_residual_autocorr_robustness.csv")
lag_autocorr.round(5)

,lag_order,aic,bic,total_acf_lag_exceedances,stable_system,multivariate_whiteness_p_value
0,1,-7.6836,-7.1170,30,True,0.0
1,2,-8.4314,-7.4852,15,True,0.0
2,3,-8.5495,-7.2222,11,True,0.0
3,4,-8.6054,-6.8953,8,True,0.0
4,5,-8.5792,-6.4849,6,True,0.0
5,6,-8.5340,-6.0540,5,True,0.0
6,7,-8.4935,-5.6261,8,True,0.0
7,8,-8.4797,-5.2235,6,True,0.0


#### VARX Residual Diagnostics

The VARX version is also checked visually and statistically. Since VARX conditions on policy/sentiment variables and crisis dummies, it is useful as a forecasting specification, but residual dependence can still remain. The same diagnostic philosophy is used as for VAR: residual time series, residual ACF plots with confidence bounds, cross-correlation patterns, heatmaps, Ljung-Box tests, ARCH tests, and stability roots.

![VARX residuals](outputs/figures/academic_varx_residual_plots.png)

![VARX residual ACF](outputs/figures/academic_varx_residual_acf.png)

![VARX residual ACF and CCF matrix](outputs/figures/academic_varx_residual_acf_ccf_matrix.png)

![VARX residual correlation](outputs/figures/academic_varx_residual_correlation_heatmap.png)

![VARX stability roots](outputs/figures/academic_varx_stability_roots.png)


In [18]:
varx_diag = pd.read_csv(TABLE_DIR / "academic_varx_residual_diagnostics.csv")
varx_acf_summary = pd.read_csv(TABLE_DIR / "academic_varx_residual_acf_summary.csv")
varx_ccf_summary = pd.read_csv(TABLE_DIR / "academic_varx_residual_ccf_summary.csv")
display(varx_diag.round(4))
display(varx_acf_summary.round(4))
display(varx_ccf_summary.sort_values("num_lags_outside_bound", ascending=False).head(12).round(4))


,variable,durbin_watson,ljung_box_p_value_lag12,white_noise_at_5pct,arch_lm_p_value_lag12,arch_effects_at_5pct,stable_system,multivariate_whiteness_p_value
0,INF,1.9973,0.1670,True,0.0000,True,True,0.0
1,UNRATE,2.0616,0.6265,True,1.0000,False,True,0.0
2,INDPRO_GROWTH,2.0693,0.0476,False,0.7251,False,True,0.0
3,M2_GROWTH,1.9982,0.3405,True,0.0000,True,True,0.0


,equation,max_abs_acf_lag_1_to_24,confidence_bound_approx,num_lags_outside_bound,any_lag_outside_bound
0,INF,0.1356,0.1018,3,True
1,UNRATE,0.0834,0.1018,0,False
2,INDPRO_GROWTH,0.1552,0.1018,2,True
3,M2_GROWTH,0.1340,0.1018,1,True


,source_residual,target_residual,max_abs_ccf_lag_1_to_24,confidence_bound_approx,num_lags_outside_bound,any_lag_outside_bound
2,M2_GROWTH,INF,0.1380,0.1018,3,True
7,UNRATE,INDPRO_GROWTH,0.1466,0.1018,3,True
4,INDPRO_GROWTH,UNRATE,0.1226,0.1018,2,True
0,UNRATE,INF,0.1159,0.1018,1,True
10,UNRATE,M2_GROWTH,0.1372,0.1018,1,True
9,INF,M2_GROWTH,0.1178,0.1018,1,True
6,INF,INDPRO_GROWTH,0.1561,0.1018,1,True
8,M2_GROWTH,INDPRO_GROWTH,0.1043,0.1018,1,True
11,INDPRO_GROWTH,M2_GROWTH,0.1029,0.1018,1,True
1,INDPRO_GROWTH,INF,0.1000,0.1018,0,False


#### Heteroskedasticity and ARCH Effects

Macroeconomic residuals often show time-varying volatility around crises. ARCH effects do not necessarily invalidate point forecasts, but they affect inference, standard errors, and confidence intervals. Modern practice often uses robust covariance estimators, bootstrap confidence intervals, stochastic-volatility models, GARCH-type extensions, or Bayesian VARs with time-varying volatility when variance instability is important.

In [19]:
var_arch = pd.read_csv(TABLE_DIR / "academic_var_arch_tests.csv")
varx_arch = pd.read_csv(TABLE_DIR / "academic_varx_arch_tests.csv")
print("VAR ARCH tests")
display(var_arch.round(4))
print("VARX ARCH tests")
display(varx_arch.round(4))


VAR ARCH tests


,equation,arch_lm_stat_lag12,arch_lm_p_value_lag12,arch_f_stat_lag12,arch_f_p_value_lag12,arch_effects_at_5pct
0,FEDFUNDS,4.1024,0.9815,0.3333,0.9828,False
1,INF,39.5670,0.0001,3.5715,0.0000,True
2,UNRATE,0.4601,1.0000,0.0370,1.0000,False
3,INDPRO_GROWTH,16.1400,0.1849,1.3573,0.1847,False
4,M2_GROWTH,115.0083,0.0000,13.5909,0.0000,True
5,SENTIMENT_CHANGE,18.8490,0.0922,1.5978,0.0903,False


VARX ARCH tests


,equation,arch_lm_stat_lag12,arch_lm_p_value_lag12,arch_f_stat_lag12,arch_f_p_value_lag12,arch_effects_at_5pct
0,INF,54.3552,0.0000,5.1445,0.0000,True
1,UNRATE,0.2330,1.0000,0.0187,1.0000,False
2,INDPRO_GROWTH,8.7385,0.7251,0.7193,0.7326,False
3,M2_GROWTH,80.8755,0.0000,8.3844,0.0000,True


### 6.4 Granger Causality

Granger causality tests whether one variable's lags contain predictive information for another variable. It is not structural causality, but it is useful for studying lead-lag relationships.

![Granger causality heatmap](outputs/figures/academic_09_granger_causality_heatmap.png)

In [20]:
granger = pd.read_csv(TABLE_DIR / "academic_granger_causality_map.csv")
granger.loc[granger["significant_at_5pct"], ["source", "target", "p_value"]].sort_values(["target", "p_value"])

,source,target,p_value
15,FEDFUNDS,INDPRO_GROWTH,2.7704e-08
17,UNRATE,INDPRO_GROWTH,1.2041e-07
18,M2_GROWTH,INDPRO_GROWTH,1.3221e-07
16,INF,INDPRO_GROWTH,5.1772e-03
5,FEDFUNDS,INF,2.9679e-03
23,INDPRO_GROWTH,M2_GROWTH,1.1178e-07
20,FEDFUNDS,M2_GROWTH,1.1183e-07
22,UNRATE,M2_GROWTH,6.0617e-06
21,INF,M2_GROWTH,7.8237e-04
10,FEDFUNDS,UNRATE,2.2961e-14


### 6.5 VARX Conditional Forecast Model

The project estimates a VARX model as the main conditional forecasting alternative to the full VAR.

- **Endogenous variables:** `INF`, `UNRATE`, `INDPRO_GROWTH`, `M2_GROWTH`.
- **Exogenous variables:** `FEDFUNDS`, `SENTIMENT_CHANGE`, `D_2008`, `D_COVID`.
- **Lag order:** 4 lags in the official baseline, retained for comparability with the VAR. The VARX lag-selection table is reported above so the reader can see whether AIC, BIC, or HQIC would favor another lag order.
- **Interpretation:** VARX does not model the future federal funds rate or sentiment internally. It forecasts the endogenous block conditional on the supplied exogenous paths. In the final 36-month evaluation below, the future exogenous paths are the observed test-period values, so the exercise should be interpreted as conditional forecasting rather than a fully real-time forecast.

This distinction is important: VAR is the stronger system model for structural interpretation, while VARX is useful for scenario analysis and conditional forecast comparison.


In [21]:
varx_sig = pd.read_csv(TABLE_DIR / "academic_varx_parameter_significance.csv")
varx_sig.query("equation == 'INF' and significant_at_5pct").sort_values("p_value").head(15).round(4)


,equation,parameter,coefficient,std_error,t_stat,p_value,lower_95,upper_95,significant_at_5pct
20,INF,L1.INF,0.4945,0.0545,9.0717,0.0000,0.3877,0.6013,True
36,INF,L2.INF,-0.2313,0.0612,-3.7815,0.0002,-0.3512,-0.1114,True
16,INF,D_COVID,0.1559,0.0450,3.4616,0.0005,0.0676,0.2442,True
48,INF,L2.M2_GROWTH,0.0978,0.0383,2.5509,0.0107,0.0227,0.1729,True
44,INF,L2.INDPRO_GROWTH,0.0502,0.0203,2.4766,0.0133,0.0105,0.0900,True
0,INF,const,0.2067,0.0880,2.3481,0.0189,0.0342,0.3793,True
12,INF,D_2008,-0.0869,0.0438,-1.9860,0.0470,-0.1727,-0.0011,True


### 6.6 Statistical Significance of Parameters

VAR and VARX models contain many lagged coefficients. Individual coefficients should be interpreted cautiously because multicollinearity across lags and equations is common. Still, t-statistics, p-values, and confidence intervals are useful for understanding the significance structure of the system.

A high number of insignificant lag coefficients may motivate restricted VARs, Bayesian shrinkage, smaller lag order, or theory-based exclusion restrictions. In this project, the full VAR is retained for system interpretation, while VARX provides a more targeted conditional-forecasting alternative.


In [22]:
var_sig_summary = pd.read_csv(TABLE_DIR / "academic_var_parameter_significance_summary.csv")
varx_sig_summary = pd.read_csv(TABLE_DIR / "academic_varx_parameter_significance_summary.csv")
display(var_sig_summary.round(4))
display(varx_sig_summary.round(4))

,equation,total_parameters,significant_at_5pct,min_p_value,share_significant_at_5pct
0,FEDFUNDS,27,9,0.0,0.3333
1,INDPRO_GROWTH,27,8,0.0,0.2963
2,INF,27,10,0.0,0.3704
3,M2_GROWTH,27,10,0.0,0.3704
4,SENTIMENT_CHANGE,27,4,0.0,0.1481
5,UNRATE,27,11,0.0,0.4074


,equation,total_parameters,significant_at_5pct,min_p_value,share_significant_at_5pct
0,INDPRO_GROWTH,21,7,0.0,0.3333
1,INF,21,7,0.0,0.3333
2,M2_GROWTH,21,9,0.0,0.4286
3,UNRATE,21,11,0.0,0.5238


In [23]:
var_sig = pd.read_csv(TABLE_DIR / "academic_var_parameter_significance.csv")
varx_sig = pd.read_csv(TABLE_DIR / "academic_varx_parameter_significance.csv")
print("Significant VAR coefficients in the inflation equation")
display(var_sig.query("equation == 'INF' and significant_at_5pct").sort_values("p_value").head(15).round(4))
print("Significant VARX coefficients in the inflation equation")
display(varx_sig.query("equation == 'INF' and significant_at_5pct").sort_values("p_value").head(15).round(4))


Significant VAR coefficients in the inflation equation


,equation,parameter,coefficient,std_error,t_stat,p_value,lower_95,upper_95,significant_at_5pct
25,INF,L1.INF,0.4675,0.0550,8.5011,0.0000,0.3597,0.5752,True
13,INF,D_COVID,0.1572,0.0440,3.5728,0.0004,0.0710,0.2434,True
61,INF,L2.INF,-0.1900,0.0620,-3.0658,0.0022,-0.3115,-0.0685,True
79,INF,L2.M2_GROWTH,0.1072,0.0386,2.7772,0.0055,0.0316,0.1829,True
1,INF,const,0.2216,0.0858,2.5821,0.0098,0.0534,0.3898,True
73,INF,L2.INDPRO_GROWTH,0.0519,0.0204,2.5446,0.0109,0.0119,0.0919,True
19,INF,L1.FEDFUNDS,0.2732,0.1091,2.5042,0.0123,0.0594,0.4871,True
151,INF,L4.M2_GROWTH,-0.0796,0.0350,-2.2712,0.0231,-0.1482,-0.0109,True
127,INF,L4.FEDFUNDS,0.2464,0.1115,2.2093,0.0272,0.0278,0.4651,True
7,INF,D_2008,-0.0923,0.0433,-2.1318,0.0330,-0.1772,-0.0074,True


Significant VARX coefficients in the inflation equation


,equation,parameter,coefficient,std_error,t_stat,p_value,lower_95,upper_95,significant_at_5pct
20,INF,L1.INF,0.4945,0.0545,9.0717,0.0000,0.3877,0.6013,True
36,INF,L2.INF,-0.2313,0.0612,-3.7815,0.0002,-0.3512,-0.1114,True
16,INF,D_COVID,0.1559,0.0450,3.4616,0.0005,0.0676,0.2442,True
48,INF,L2.M2_GROWTH,0.0978,0.0383,2.5509,0.0107,0.0227,0.1729,True
44,INF,L2.INDPRO_GROWTH,0.0502,0.0203,2.4766,0.0133,0.0105,0.0900,True
0,INF,const,0.2067,0.0880,2.3481,0.0189,0.0342,0.3793,True
12,INF,D_2008,-0.0869,0.0438,-1.9860,0.0470,-0.1727,-0.0011,True


## 7. Model Quality Evaluation and Testing

Forecast quality is evaluated out-of-sample on the final 36 months. RMSE penalizes large forecast errors more strongly, while MAE is easier to interpret as an average absolute error. Because different models use different forecast designs, the forecast-design column should be read carefully.

In [24]:
econ_metrics = pd.read_csv(TABLE_DIR / "academic_econometric_forecast_metrics.csv")
econ_metrics.sort_values("rmse").round(4)

,model,target,forecast_design,rmse,mae
2,Random Walk (direct 36-month),inf,final_36_months,0.1883,0.1515
0,VAR,inf,final_36_months,0.1991,0.1696
1,VARX,inf,final_36_months,0.2176,0.1831


![Econometric forecast comparison](outputs/figures/academic_12_econometric_forecast_comparison.png)

### 7.1 Rolling Multi-Step Forecast Robustness

A single test split can be sensitive to one period. The rolling exercise re-estimates VARX from each valid origin beginning in 2024 and forecasts 3 months ahead. This gives a more stable comparison against a random-walk benchmark.

In [25]:
rolling = pd.read_csv(TABLE_DIR / "academic_rolling_3month_varx_rmse.csv")
rolling.round(4)

,variable,varx_rmse,random_walk_rmse,winner
0,INDPRO_GROWTH,0.5225,0.7226,VARX
1,INF,0.1693,0.1781,VARX
2,M2_GROWTH,0.2433,0.1820,Random Walk
3,UNRATE,0.1762,0.1138,Random Walk


![Rolling VARX RMSE](outputs/figures/academic_13_rolling_varx_rmse.png)

## 8. Model Application: IRF, FEVD, and Shock Analysis

After estimating the VAR, the model can be used for economic interpretation. The Cholesky identification imposes a recursive ordering, so the results are conditional on that ordering. IRFs show how variables respond dynamically to shocks; FEVD shows which shocks explain forecast uncertainty.

### 8.1 Impulse Response Functions

Impulse Response Functions are central to the policy interpretation of the project. They trace how a one-standard-deviation structural shock propagates through the macroeconomic system over time.

The IRFs use recursive Cholesky identification. This means that contemporaneous causal ordering is imposed rather than estimated. The ordering is economically interpretable but debatable; therefore, IRFs should be read as conditional policy-scenario evidence rather than definitive structural truth.

Important interpretation points:

- **Monetary policy shock:** a federal funds rate shock captures a tightening in short-term policy conditions. The response of inflation, unemployment, output growth, money growth, and sentiment indicates the transmission path.
- **Inflation shock:** shows whether price shocks persist and whether policy/labor/output variables react dynamically.
- **Unemployment shock:** captures labor-market slack and its propagation to inflation and real activity.
- **Output and money shocks:** help evaluate demand-side and liquidity channels.
- **Persistence:** responses that decay slowly suggest long-lived macroeconomic propagation.

The plotted IRFs include confidence intervals, so responses whose bands include zero should be interpreted cautiously.

![Structural IRF](outputs/figures/academic_10_structural_irf.png)

In [26]:
policy_irf = pd.read_csv(TABLE_DIR / "academic_irf_monetary_policy_shock.csv")
policy_irf[["response", "impact_h0", "response_h3", "response_h6", "response_h12", "cumulative_response_h0_to_h24", "peak_abs_response", "peak_abs_response_horizon"]].round(4)

,response,impact_h0,response_h3,response_h6,response_h12,cumulative_response_h0_to_h24,peak_abs_response,peak_abs_response_horizon
0,FEDFUNDS,0.1249,0.2708,0.3292,0.3425,7.1259,0.3518,9
1,INF,0.0483,-0.0182,-0.0010,-0.0084,-0.0237,0.0629,1
2,UNRATE,-0.0126,-0.2278,-0.1953,-0.1863,-4.3062,-0.2673,1
3,INDPRO_GROWTH,0.1263,-0.0436,-0.0030,-0.0107,0.2802,0.3659,1
4,M2_GROWTH,-0.1086,-0.0665,-0.0291,-0.0132,-0.8214,-0.1787,1
5,SENTIMENT_CHANGE,0.7289,0.0485,-0.0455,-0.0024,1.1001,0.7289,0


In [27]:
irf_table = pd.read_csv(TABLE_DIR / "academic_irf_interpretation_table.csv")
irf_table.query("response == 'INF'").sort_values("peak_abs_response", key=lambda s: s.abs(), ascending=False).head(10).round(4)

,response,shock,impact_h0,response_h3,response_h6,response_h12,response_h24,cumulative_response_h0_to_h24,peak_abs_response,peak_abs_response_horizon,persistence_ratio_h12_to_impact
7,INF,INF,0.2310,-0.0056,0.0081,0.0011,0.0004,0.3840,0.2310,0,0.0047
6,INF,FEDFUNDS,0.0483,-0.0182,-0.0010,-0.0084,-0.0061,-0.0237,0.0629,1,-0.1737
9,INF,INDPRO_GROWTH,0.0000,0.0265,0.0048,-0.0024,-0.0024,0.0509,0.0380,2,NaN
10,INF,M2_GROWTH,0.0000,0.0264,0.0054,0.0019,0.0002,0.0640,0.0264,3,NaN
11,INF,SENTIMENT_CHANGE,0.0000,-0.0114,0.0120,0.0009,-0.0004,0.0031,-0.0211,4,NaN
8,INF,UNRATE,0.0000,-0.0038,-0.0068,0.0007,0.0011,-0.0032,0.0071,4,NaN


### 8.2 Forecast Error Variance Decomposition

FEVD complements IRFs by asking a different question: what share of forecast uncertainty in each variable is attributable to each structural shock?

Short-horizon FEVD is often dominated by the variable's own shock. At longer horizons, other shocks may become more important, revealing cross-variable transmission. For inflation, the table below shows whether inflation uncertainty is mainly self-driven or increasingly explained by policy, labor-market, output, money, or sentiment shocks.

![FEVD](outputs/figures/academic_11_fevd.png)

In [28]:
fevd = pd.read_csv(TABLE_DIR / "academic_fevd_selected_horizons.csv")
fevd.query("response == 'INF'").pivot(index="horizon", columns="shock", values="variance_share").round(3)

shock,FEDFUNDS,INDPRO_GROWTH,INF,M2_GROWTH,SENTIMENT_CHANGE,UNRATE
horizon,,,,,,
1,0.042,0.000,0.958,0.000,0.000,0.000
3,0.100,0.021,0.864,0.009,0.004,0.001
6,0.104,0.030,0.835,0.019,0.012,0.001
12,0.106,0.030,0.828,0.019,0.014,0.002
24,0.114,0.031,0.820,0.019,0.014,0.002


In [29]:
dominant_fevd = pd.read_csv(TABLE_DIR / "academic_fevd_dominant_shocks.csv")
dominant_fevd.round(4)

,response,horizon,shock,variance_share
0,FEDFUNDS,1,FEDFUNDS,1.0000
1,FEDFUNDS,3,FEDFUNDS,0.9729
2,FEDFUNDS,6,FEDFUNDS,0.9369
3,FEDFUNDS,12,FEDFUNDS,0.9083
4,FEDFUNDS,24,FEDFUNDS,0.8880
5,INDPRO_GROWTH,1,UNRATE,0.4985
6,INDPRO_GROWTH,3,UNRATE,0.4437
7,INDPRO_GROWTH,6,UNRATE,0.4495
8,INDPRO_GROWTH,12,UNRATE,0.4513
9,INDPRO_GROWTH,24,UNRATE,0.4509


## 9. Machine Learning Comparison

Machine learning models are estimated on lagged macroeconomic features to forecast inflation. The comparison includes:

- Ridge Regression as a regularized linear baseline.
- Random Forest for nonlinear interactions and threshold behavior.
- Gradient Boosting for sequential nonlinear error correction.
- Random Walk as a benchmark.

ML models can improve pure forecasting performance, but they provide weaker economic structure than VAR and VARX. They do not naturally produce IRFs, FEVD, Granger-causality interpretation, or structural shock analysis. For this reason, ML models are treated as forecast benchmarks rather than replacements for the econometric policy-analysis framework.


In [30]:
ml_metrics = pd.read_csv(TABLE_DIR / "academic_ml_forecast_metrics.csv")
ml_metrics.round(4)

,model,target,forecast_design,rmse,mae
0,Ridge Regression,inf,one_step_lagged_features_final_36_months,0.1595,0.1168
1,Random Walk (one-step),inf,one_step_lagged_features_final_36_months,0.1739,0.1320
2,Random Forest,inf,one_step_lagged_features_final_36_months,0.1794,0.1303
3,Gradient Boosting,inf,one_step_lagged_features_final_36_months,0.1836,0.1390


![ML forecast comparison](outputs/figures/academic_14_ml_forecast_comparison.png)

In [31]:
ranking = pd.read_csv(TABLE_DIR / "academic_all_model_forecast_ranking.csv")
ranking.round(4)

,model,target,forecast_design,rmse,mae
0,Ridge Regression,inf,one_step_lagged_features_final_36_months,0.1595,0.1168
1,Random Walk (one-step),inf,one_step_lagged_features_final_36_months,0.1739,0.1320
2,Random Forest,inf,one_step_lagged_features_final_36_months,0.1794,0.1303
3,Gradient Boosting,inf,one_step_lagged_features_final_36_months,0.1836,0.1390
4,Random Walk (direct 36-month),inf,final_36_months,0.1883,0.1515
5,VAR,inf,final_36_months,0.1991,0.1696
6,VARX,inf,final_36_months,0.2176,0.1831


## 10. Final Discussion and Limitations

The project addresses the original problem by building a macroeconomic forecasting system and comparing interpretable econometric methods with machine-learning benchmarks.

Main empirical conclusions:

- CPI, M2, and consumer sentiment are non-stationary in levels; transformed variables are stationary.
- Pairwise Engle-Granger tests do not support cointegration among the non-stationary level variables, so differenced/growth-rate modeling is appropriate here.
- AIC selects a VAR with 4 monthly lags. Lag robustness shows that higher lag orders reduce residual ACF exceedances, but at the cost of additional parameters.
- The VAR and VARX systems are stable, but visual residual ACF/CCF diagnostics show that some residual dependence remains. This is normal in real macroeconomic data but must be reported.
- ARCH tests indicate heteroskedasticity in selected equations, especially inflation and M2 growth. This means inference and IRF confidence intervals should be interpreted with caution.
- The federal funds rate has significant Granger-predictive content for inflation.
- VARX beats random walk for inflation in rolling 3-month forecasts from 2024 onward.
- In the final 36-month one-step ML comparison, Ridge Regression performs best, showing that regularized lagged linear models can be competitive.
- Econometric models remain stronger for interpretation because they support Granger causality, IRF, FEVD, and system-level economic analysis.

Methodological limitations and possible improvements:

- Granger causality is predictive, not structural causality.
- Cholesky IRFs depend on variable ordering; robustness to alternative orderings should be considered.
- Residual autocorrelation could be reduced through alternative lag length, seasonal controls, richer exogenous variables, or restricted/Bayesian VAR models.
- Heteroskedasticity suggests considering robust covariance estimators, bootstrap IRF intervals, GARCH-type volatility modeling, or stochastic-volatility VARs.
- VAR parameters may be unstable across regimes; rolling-window estimation or time-varying parameter VARs would be stronger.
- Exogenous variables in VARX forecasts are conditional assumptions; true real-time forecasting would require forecasting those variables too.
- Important omitted variables include oil prices, exchange rates, fiscal policy, financial conditions, and inflation expectations.
- Machine-learning results are sensitive to feature engineering and the small size of monthly macroeconomic datasets.


## 11. Reproducibility

Run the base FRED download and baseline analysis:

```python
%run src/macro_time_series_analysis.py
```

Run the full academic econometrics and ML pipeline:

```python
%run src/advanced_macro_var_analysis.py
```

Run the interactive dashboard:

```bash
streamlit run dashboard_app.py
```

The notebook contains the official fixed baseline specification for report consistency. The dashboard acts as a sensitivity-analysis tool where train/test split, variables, model type, and lag order can be changed interactively.

The project writes all reproducible inputs, tables, and figures to `data/` and `outputs/`.
